# Tyche Data API — data discovery & sector analysis

This notebook is a guided tour of the **discovery and bulk-price** parts of the
`tyche-client` SDK (v0.3+):

| Step | Method |
|------|--------|
| Discover what data exists | `client.sources()`, `client.search()` |
| List indexes & members | `client.indices()`, `client.constituents()` |
| Build a classified ticker universe | `client.universe()` |
| Pull full OHLCV for many tickers | `client.ohlcv()` |
| Index → filtered constituents → prices, in one call | `client.index_prices()` |

We finish with a **worked example**: pull every Energy name in the S&P 500,
measure sector breadth, dispersion, and how the basket tracks crude oil.

> The older `example.ipynb` covers the single-ticker / price-matrix / macro basics.
> This one focuses on the bulk-pull workflow.

---
## 0. Setup

Mint a read-only key in the web app (**Settings → API keys**) or via
`POST /me/api-keys/`, then paste it below. The key is shown only once.

In [ ]:
import os

from tyche_client import TycheClient

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

load_dotenv(override=True)  # reads TYCHE_BASE_URL / TYCHE_API_KEY from a local .env file (see .env.example)
# override=True: without it, editing .env and re-running this cell in an already-running
# kernel silently keeps the OLD value, since load_dotenv() won't clobber an env var that
# a previous run of this same cell already set -- a real bug this caused once, not hypothetical.

client = TycheClient(
    os.getenv("TYCHE_BASE_URL", "http://localhost:8000"),
    os.getenv("TYCHE_API_KEY", "tyk_paste_your_key_here"),
)
client

Matplotlib is building the font cache; this may take a moment.


## 1. What data is available?

`sources()` returns a small reference table of every *kind* of series the API can
serve, with an example id and the SDK method to reach it. Start here when you don't
yet know what to ask for.

In [2]:
client.sources()

TycheApiError: [401] Could not validate credentials

### Searching the catalog

`search(q)` ranks matching tickers and economic series across every source. Pass
`source=` to restrict the search — e.g. only FRED macro series, or only equities.

In [ ]:
# Free-text search across everything (equities, indexes, macro series...)
client.search("apple")

In [ ]:
# Narrow to a single source. Useful when a ticker and a macro series share letters.
client.search("oil", source="eia")

## 2. Indexes and their constituents

`indices()` lists the official indexes we track (with live member counts).
`constituents(slug)` returns the current members; you can pre-filter by `sector`
or `exchange` on the server so you only pull what you need.

In [ ]:
client.indices()

In [ ]:
# All current S&P 500 members classified as Energy.
energy_members = client.constituents("sp500", sector="Energy")
print(f"{len(energy_members)} Energy constituents")
energy_members[["ticker", "company_name", "sector", "exchange"]].head(10)

## 3. A classified universe (sector **and** industry **and** country)

Index constituents carry a `sector` but **not** an `industry` — industry only exists
in the fundamentals snapshot. `universe()` joins that in and also derives a 2-letter
`country` code from each ticker's exchange suffix (`.TO`→CA, `.T`→JP, bare→US, …).

Filter dimensions: `index` (scope to an index), `sector`, `industry`, `country`,
`exchange`, `market_cap_min`.

In [ ]:
# Every Technology name in the S&P 500, with industry + market cap attached.
tech = client.universe(index="sp500", sector="Technology")
tech.sort_values("market_cap", ascending=False).head(10)

### Country filter

The universe isn't limited to one index — you can sweep the whole classified set by
country and size. Here: Canadian-listed names above a \$1B market cap.

In [ ]:
ca_large = client.universe(country="CA", market_cap_min=1e9)
ca_large[["ticker", "name", "sector", "industry", "country", "market_cap"]].head(10)

### Industry filter and the coverage warning

Industry classification is sparse for illiquid / non-US names. When you filter by
`industry`, any ticker with no classification is **excluded**, and the SDK emits a
`UserWarning` telling you how many were dropped — so a thin result never passes
silently.

In [ ]:
import warnings

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    semis = client.universe(index="sp500", industry="Semiconductors")
    for w in caught:
        print("warning:", w.message)

semis[["ticker", "name", "industry", "market_cap"]]

## 4. Bulk OHLCV — the long/tidy frame

`ohlcv(tickers, ...)` returns **one row per (ticker, date)** with `Open, High, Low,
Close, Volume` columns. This tidy shape is easy to `groupby` and `pivot`. Large
ticker lists are chunked automatically under the API's 50-per-call cap, so you can
hand it a whole index.

Pass `fields=[...]` to keep only the columns you need in the returned frame. The API still sends full OHLCV bars.

In [ ]:
bars = client.ohlcv(["AAPL", "MSFT", "NVDA"], start="2024-01-01")
bars.head()

In [ ]:
# Tidy → wide is a one-liner when you want a single field aligned across tickers.
closes = bars.pivot(index="date", columns="ticker", values="Close")
closes.tail()

### One call: index → filtered constituents → prices

`index_prices()` composes steps 2–4. Give it an index slug plus optional
`sector` / `industry` / `country` filters and it resolves the tickers (via
`universe()` when an industry/country filter is present, otherwise the constituent
list) and returns the same long/tidy OHLCV frame.

In [ ]:
energy = client.index_prices("sp500", sector="Energy", start="2023-01-01")
print(f"{energy['ticker'].nunique()} tickers, {len(energy):,} rows")
energy.head()

---
## 5. Worked example — S&P 500 Energy sector

Using the `energy` frame we just pulled, we'll look at:

1. **Cumulative performance** of every name (normalized to the start date).
2. An **equal-weight sector basket** and its **breadth** (share of names up YTD).
3. **Cross-sectional dispersion** — how differently the names are moving.
4. **Best / worst performers** over the window.
5. How the basket **co-moves with crude oil** (a FRED macro series).

In [ ]:
# Wide close matrix, then simple daily returns.
px  = energy.pivot(index="date", columns="ticker", values="Close").sort_index()
# Drop names with little history so the basket isn't distorted by late listings.
px  = px.dropna(axis=1, thresh=int(0.9 * len(px)))
ret = px.pct_change().dropna(how="all")
print(f"{px.shape[1]} names with full history over {px.index[0].date()} → {px.index[-1].date()}")

In [ ]:
# 1. Normalized cumulative price of each name (spaghetti plot).
norm = px / px.iloc[0]
ax = norm.plot(figsize=(11, 5), legend=False, alpha=0.5, color="steelblue")
ax.set_title("S&P 500 Energy — cumulative price (normalized to 1.0)")
ax.axhline(1.0, color="black", lw=0.8)
ax.set_ylabel("growth of 1.0");

In [ ]:
# 2. Equal-weight basket vs. breadth (fraction of names above their start price).
basket = (1 + ret.mean(axis=1)).cumprod()
breadth = (norm > 1.0).mean(axis=1)

fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True,
                             gridspec_kw={"height_ratios": [2, 1]})
basket.plot(ax=a1, color="darkgreen")
a1.set_title("Equal-weight Energy basket"); a1.set_ylabel("growth of 1.0"); a1.axhline(1, color="k", lw=.8)
breadth.plot(ax=a2, color="gray")
a2.set_title("Breadth — share of names above start price"); a2.set_ylim(0, 1)
a2.axhline(0.5, color="red", lw=.8, ls="--"); plt.tight_layout();

In [ ]:
# 3. Cross-sectional dispersion: std of same-day returns across names (annualized).
dispersion = ret.std(axis=1) * np.sqrt(252)
ax = dispersion.rolling(21).mean().plot(figsize=(11, 4), color="purple")
ax.set_title("Cross-sectional dispersion (21d avg, annualized)")
ax.set_ylabel("stdev of daily returns");

In [ ]:
# 4. Best / worst total return over the window.
total_ret = (px.iloc[-1] / px.iloc[0] - 1).sort_values()
ranked = pd.concat([total_ret.head(5), total_ret.tail(5)])
ax = ranked.plot.barh(figsize=(8, 5),
                      color=["firebrick"] * 5 + ["seagreen"] * 5)
ax.set_title("Worst 5 / best 5 total return"); ax.set_xlabel("total return")
ax.axvline(0, color="black", lw=0.8);

### 5. Does the basket track crude oil?

We pull spot WTI from FRED (`DCOILWTICO`) with `series()`, align it to our trading
days, and compare returns. This mixes the **equity bulk-pull** path with the
**single macro-series** path in one analysis.

In [ ]:
wti = client.series("fred", "DCOILWTICO")          # WTI spot, $/bbl
wti = wti[wti.index >= px.index[0]]

# Align both to the equity trading calendar and take daily returns.
oil_ret    = wti.reindex(px.index).ffill().pct_change()
basket_ret = ret.mean(axis=1)
aligned    = pd.concat([basket_ret.rename("energy_basket"),
                        oil_ret.rename("wti")], axis=1).dropna()

corr = aligned["energy_basket"].corr(aligned["wti"])
print(f"Daily-return correlation, Energy basket vs WTI: {corr:.2f}")

cum = (1 + aligned).cumprod()
ax = cum.plot(figsize=(11, 5))
ax.set_title(f"Energy basket vs WTI (daily-return corr = {corr:.2f})")
ax.set_ylabel("growth of 1.0"); ax.axhline(1, color="k", lw=.8);

### Intra-sector correlation

Finally, how tightly do the Energy names move together? A high average pairwise
correlation means the sector trades as one factor (little diversification within it).

In [ ]:
cmat = ret.corr()
avg_pairwise = cmat.where(~np.eye(len(cmat), dtype=bool)).stack().mean()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cmat, vmin=-1, vmax=1, cmap="RdBu_r")
ax.set_xticks(range(len(cmat))); ax.set_xticklabels(cmat.columns, rotation=90, fontsize=7)
ax.set_yticks(range(len(cmat))); ax.set_yticklabels(cmat.index, fontsize=7)
ax.set_title(f"Energy return correlations (avg pairwise = {avg_pairwise:.2f})")
fig.colorbar(im, fraction=0.046, pad=0.04); plt.tight_layout();

---
## Notes & gotchas

- **Long/tidy is the default** for multi-ticker OHLCV (`ohlcv`, `index_prices`).
  `pivot(index="date", columns="ticker", values="Close")` gives you a wide matrix
  when you want one aligned field.
- **Auto-chunking:** ticker lists longer than the API's 50-per-call cap are split and
  reassembled for you — pass a whole index without worrying about it.
- **Industry coverage is partial.** Filtering `universe(industry=...)` drops
  unclassified tickers and warns with the count. Sector/country are more complete.
- **Country** is inferred from the ticker suffix (`US`, `CA`, `JP`, `KR`, `TW`, `GB`,
  `DE`, `FR`, …), not a stored field.
- Prices only exist for **valid trading days**; there are no weekend/holiday rows, so
  `reindex(...).ffill()` is the right way to align a macro series onto the equity
  calendar (as we did with WTI).